In [0]:
from pyspark.sql.functions import col, when
import sys
sys.path.insert(0, '/Workspace/Users/ragha.raj90@gmail.com/ipl_analysis/src')

from schema.silver import *

In [0]:
match_metadata_table = "bootcamp_students.zachy_ragha_raj90.ipl_matches"
ball_by_ball_rable = "bootcamp_students.zachy_ragha_raj90.ipl_deliveries"

In [0]:
match_meta_df = spark.table(match_metadata_table)
ball_df = spark.table(ball_by_ball_rable)

In [0]:
match_df = match_meta_df.select(
    col("match_id").alias("match_id"),
    col("teams")[0].alias("team_1"),
    col("teams")[1].alias("team_2"),
    col("dates")[0].alias("match_date"),
    col("season").alias("season_year"),
    col("city").alias("venue_name"),
    col("toss_winner").alias("toss_winner"),
    col("outcome_winner").alias("match_winner"),
    when(col("runs") != None, "Runs")
        .when((col("wickets") != None), "Wickets")
        .otherwise("Unknown")
        .alias("win_type"),
    when(col("win_type") != 'Unknown', "Result")
        .otherwise("Unknown")
        .alias("outcome_type"),
    col("player_of_match")[0].alias("manofmach"),
    when(col("runs") != None, col("runs"))
    .when((col("wickets") != None), col("wickets"))
    .otherwise(0)
    .alias("win_margin"),
    )
match_df.write.mode("overwrite").saveAsTable("bootcamp_students.zachy_ragha_raj90.silver_matches")

In [0]:
teams_df = match_df.select("team_1").distinct()
teams_df.write.mode("overwrite").saveAsTable("bootcamp_students.zachy_ragha_raj90.silver_teams")

In [0]:
ball_by_ball_df = ball_df.select(
    col("match_id").alias("match_id"),
    col("innings_no").alias("innings_no"),
    col("team").alias("batting_team"),
    when(col("team") == col("team_1"), col("team_2"))
    .otherwise(col("team_1"))
    .alias("bowling_team"),
    col("over_id").alias("over_id"),
    col("ball").alias("ball_id"),
    col("deliveries.batter").alias("batter"),
    col("deliveries.bowler").alias("bowler"),
    col("deliveries.non_striker").alias("non_striker"),
)

In [0]:
ball_by_ball_df.filter(col("innings_no") == 1).show(5)